# Engineering Source Extraction

Reusable runner for structured engineering source records.

Change only the configuration cell. Source-specific evidence is maintained in
`tools/source_extractors/`. The notebook validates the selected extractor,
writes canonical outputs, creates one ZIP, and downloads it automatically in
Google Colab.


## 1. Configuration

In [ ]:
from pathlib import Path

SOURCE_ID = "SOURCE_01"
SOURCE_FILENAME = "SOURCE_01_bismuth_microstructure.yaml"
PAPER_TITLE = (
    "Microstructure analysis of bismuth absorbers for "
    "transition-edge sensor X-ray microcalorimeters"
)
PDF_FILENAME = "1711.02215v1.pdf"

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None


## 2. Locate repository and import pipeline

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def bootstrap_repo() -> Path:
    candidates: list[Path] = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])
    candidates.extend(
        [
            Path("/content/sensors-becker"),
            Path("/home/dan/sensors-becker"),
            Path.home() / "sensors-becker",
        ]
    )

    for candidate in candidates:
        if (
            candidate.is_dir()
            and (candidate / "engineering_navigator").is_dir()
            and (candidate / "tools").is_dir()
        ):
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            import subprocess
            subprocess.run(
                ["git", "clone", REPOSITORY_URL, str(target)],
                check=True,
            )
        return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. "
        "Set REPO_ROOT_OVERRIDE to its absolute path."
    )


REPO_ROOT = bootstrap_repo()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.engineering_source_record import (
    SourceRecordError,
    build_export_package,
    build_source_paths,
    download_in_colab,
    load_source_record,
    validate_completed_record,
    write_completed_record,
)
from tools.source_extractors import extract_source

PATHS = build_source_paths(
    REPO_ROOT,
    source_filename=SOURCE_FILENAME,
    source_id=SOURCE_ID,
)

print(f"Repository : {PATHS.repo_root}")
print(f"Source     : {PATHS.source_record.relative_to(PATHS.repo_root)}")
print(f"Paper      : {PAPER_TITLE}")
print(f"PDF label  : {PDF_FILENAME}")
print(f"Export ZIP : {PATHS.export_zip.relative_to(PATHS.repo_root)}")


## 3. Load canonical source-record scaffold

In [ ]:
scaffold = load_source_record(PATHS.source_record)

if scaffold["source_id"] != SOURCE_ID:
    raise SourceRecordError(
        f"Configuration selects {SOURCE_ID}, but YAML contains "
        f"{scaffold['source_id']}"
    )

print(f"Loaded: {scaffold['title']}")
print(f"Status: {scaffold['extraction_status']}")


## 4. Run registered extractor

In [ ]:
completed_record = extract_source(SOURCE_ID, scaffold)

print(f"Materials                : {len(completed_record['materials'])}")
print(f"Fabrication methods      : {len(completed_record['fabrication_methods'])}")
print(f"Design variables         : {len(completed_record['design_variables'])}")
print(f"Reported values          : {len(completed_record['reported_values'])}")
print(f"Engineering relationships: {len(completed_record['engineering_relationships'])}")


## 5. Validate

In [ ]:
validation_errors = validate_completed_record(completed_record)

if validation_errors:
    raise SourceRecordError(
        "Validation failed:\n- " + "\n- ".join(validation_errors)
    )

print("Validation: PASS")


## 6. Write outputs

In [ ]:
written_files = write_completed_record(completed_record, PATHS)

for label, path in written_files.items():
    print(f"{label:32} {path.relative_to(PATHS.repo_root)}")


## 7. Build and download export ZIP

In [ ]:
zip_path = build_export_package(written_files, PATHS)

print(f"Export package: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")

if not download_in_colab(zip_path):
    print("Automatic download is available only in Google Colab.")


## 8. Engineering handoff

The exported source record is ready for multi-source comparison.

To process another registered source, change `SOURCE_ID` and
`SOURCE_FILENAME`, then rerun from the top.

*Admissible generalizations trail leading specifications.*
